# RAG Pipeline — E-Commerce Customer Support Assistant

This notebook builds and evaluates the full Retrieval-Augmented Generation (RAG) pipeline:
load & clean documents → chunk → embed → store in a vector database → retrieve → prompt →
generate with a local Ollama LLM → evaluate.

**Domain:** E-commerce customer support (orders, shipping, returns, refunds, payment, account,
cancellation, delivery, FAQs, products).

The persisted vector store produced at the end of this notebook is loaded directly by the
FastAPI backend (`backend/app/services/retrieval.py`) — the backend never rebuilds it.

## 0. Setup

Install/verify dependencies (run once).

In [1]:
# !pip install pandas numpy faiss-cpu sentence-transformers ollama python-dotenv

import os
import glob
import json
import textwrap
from pathlib import Path

import pandas as pd

DOCS_DIR = Path("../data/documents")
VECTOR_STORE_DIR = Path("../backend/data/vector_store")
COLLECTION_NAME = "ecommerce_support"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print("Docs dir:", DOCS_DIR.resolve())
print("Vector store output dir:", VECTOR_STORE_DIR.resolve())


Docs dir: E:\rag-ecommerce-assistant\data\documents
Vector store output dir: E:\rag-ecommerce-assistant\backend\data\vector_store


## 2.1 Load & Inspect

We load every Markdown file in `data/documents/`. Each file corresponds to one support topic
(orders, shipping, returns, refunds, payment, account, cancellation, delivery, FAQs, products).

In [2]:
doc_paths = sorted(glob.glob(str(DOCS_DIR / "*.md")))

documents = []
for path in doc_paths:
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    documents.append({"source": Path(path).name, "text": text})

df_docs = pd.DataFrame(documents)
df_docs["n_chars"] = df_docs["text"].str.len()
df_docs["n_words"] = df_docs["text"].str.split().apply(len)
df_docs[["source", "n_chars", "n_words"]]


,source,n_chars,n_words
0,account.md,1091,191
1,cancellation.md,1070,188
2,delivery.md,1194,198
3,faqs.md,1090,176
4,orders.md,1481,262
5,payment.md,1173,196
6,products.md,1336,227
7,refunds.md,1136,201
8,returns.md,1240,215
9,shipping.md,1446,241


**Inspection notes:**

- **How many documents?** 10 Markdown files, one per support topic.
- **What formats?** Plain Markdown (`.md`) text — all UTF-8, all directly text-extractable (no
  scanning/OCR needed since they were authored as text, not images).
- **Which files failed to parse or need OCR?** None. Every file loaded cleanly with `open().read()`
  and every file has non-zero length, confirmed by the table above.
- Each document uses `##` headers to separate individual Q&A entries, which we exploit as a
  natural chunk boundary in the next section.

## 2.2 Chunking Strategy

**Strategy chosen: section-based chunking**, splitting on each `## ` markdown header so that every
chunk is exactly one self-contained Q&A entry (question + answer), rather than a fixed-size window
that could cut a question off from its answer.

**Justification:** Fixed-size chunking (e.g. 500 characters with 50-character overlap) is simple,
but for FAQ-style documents like ours it risks splitting a question from its answer, or merging two
unrelated Q&A pairs into one chunk — both hurt retrieval precision. Since every document in this
domain is already naturally segmented into short, self-contained Q&A sections (each 80-250 words),
section-based chunking preserves exactly one complete idea per chunk with no information loss and
no arbitrary overlap parameter to tune. We keep a small `MAX_CHUNK_CHARS` safety cap and fall back
to a fixed-size split with overlap only for the rare section that exceeds it.

In [3]:
MAX_CHUNK_CHARS = 1200      # safety cap for any single chunk
FALLBACK_OVERLAP = 100      # overlap used only if a section exceeds the cap

def split_into_sections(text: str) -> list[str]:
    """Split a markdown document on '## ' headers; each section = one Q&A entry."""
    parts = text.split("\n## ")
    sections = []
    for i, part in enumerate(parts):
        part = part.strip()
        if not part:
            continue
        # Re-add the '## ' header marker we split on (except the very first, top-level '# ' title)
        if i > 0:
            part = "## " + part
        sections.append(part)
    return sections

def fixed_size_split(text: str, size: int, overlap: int) -> list[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

chunks = []
for doc in documents:
    sections = split_into_sections(doc["text"])
    for idx, section in enumerate(sections):
        if len(section) <= MAX_CHUNK_CHARS:
            chunks.append({"source": doc["source"], "chunk_index": idx, "text": section})
        else:
            for sub_idx, sub in enumerate(fixed_size_split(section, MAX_CHUNK_CHARS, FALLBACK_OVERLAP)):
                chunks.append({
                    "source": doc["source"],
                    "chunk_index": f"{idx}.{sub_idx}",
                    "text": sub,
                })

df_chunks = pd.DataFrame(chunks)
df_chunks["chunk_id"] = df_chunks.apply(lambda r: f"{r['source']}::{r['chunk_index']}", axis=1)
df_chunks["n_chars"] = df_chunks["text"].str.len()

print(f"Total chunks: {len(df_chunks)}")
df_chunks[["chunk_id", "n_chars"]].head(10)


Total chunks: 67


,chunk_id,n_chars
0,account.md::0,9
1,account.md::1,155
2,account.md::2,194
3,account.md::3,159
4,account.md::4,192
5,account.md::5,181
6,account.md::6,188
7,cancellation.md::0,14
8,cancellation.md::1,192
9,cancellation.md::2,186


## 2.3 Embeddings & Vector Store

We embed every chunk with `sentence-transformers/all-MiniLM-L6-v2` (fast, 384-dim, strong for short
FAQ-style text) and persist the vectors to disk in a FAISS index, so the FastAPI backend can
load the store directly without recomputing embeddings at request time.


In [4]:
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

# Normalize embeddings so that inner product == cosine similarity
embeddings = embedding_model.encode(
    df_chunks["text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True,
)
embeddings = np.asarray(embeddings, dtype="float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

# FAISS only stores vectors, so we persist the chunk text/metadata alongside it,
# ordered so that FAISS row i <-> chunk_records[i].
chunk_records = [
    {
        "chunk_id": cid,
        "text": text,
        "source": source,
        "chunk_index": str(chunk_index),
    }
    for cid, text, source, chunk_index in zip(
        df_chunks["chunk_id"], df_chunks["text"], df_chunks["source"], df_chunks["chunk_index"]
    )
]

index_path = VECTOR_STORE_DIR / f"{COLLECTION_NAME}.faiss"
meta_path = VECTOR_STORE_DIR / f"{COLLECTION_NAME}.pkl"

faiss.write_index(index, str(index_path))
with open(meta_path, "wb") as f:
    pickle.dump(chunk_records, f)

print(f"Persisted {index.ntotal} chunks to FAISS at: {index_path.resolve()}")


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Persisted 67 chunks to FAISS at: E:\rag-ecommerce-assistant\backend\data\vector_store\ecommerce_support.faiss


## 2.4 Retrieval & Prompting

We implement a `retrieve()` function and test it against **10+ sample questions** covering every
document topic. We then build the prompt template that combines retrieved context with the user's
question, with citation-style grounding (`[1]`, `[2]`, ...) back to the source document.

In [5]:
def retrieve(question: str, top_k: int = 4) -> list[dict]:
    query_vec = embedding_model.encode([question], normalize_embeddings=True)
    query_vec = np.asarray(query_vec, dtype="float32")
    scores, indices = index.search(query_vec, top_k)

    out = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        rec = chunk_records[idx]
        out.append({
            "chunk_id": rec["chunk_id"],
            "text": rec["text"],
            "source": rec.get("source", "unknown"),
            "score": float(score),
        })
    return out


SYSTEM_PROMPT = """You are a helpful customer support assistant for an e-commerce store.
Answer the user's question using ONLY the information in the Context section below.
If the answer is not contained in the context, say clearly that you don't have that information
and suggest the user contact human support. Do NOT make anything up. Keep answers concise, and
mention which source number(s) you used, e.g. [1]."""


def build_prompt(question: str, chunks: list[dict]) -> str:
    context = "\n\n".join(f"[{i+1}] (source: {c['source']})\n{c['text']}" for i, c in enumerate(chunks))
    return f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer using only the context above, citing source numbers."


# Quick sanity check
sample_chunks = retrieve("Where is my order?", top_k=3)
for c in sample_chunks:
    print(f"[{c['score']:.3f}] {c['chunk_id']}")


[0.534] orders.md::3
[0.489] orders.md::1
[0.484] orders.md::0


In [6]:
TEST_QUESTIONS = [
    "Where is my order?",
    "How long do I have to return an item?",
    "How much does express shipping cost?",
    "When will I get my refund after a return?",
    "Can I cancel my order after it has shipped?",
    "What payment methods can I use?",
    "How do I reset my forgotten password?",
    "What happens if my package arrives damaged?",
    "Do you ship to countries outside the US?",
    "How do I know if a product is in stock?",
    "Can I get store credit instead of a refund?",
    "Is it safe to save my credit card on your site?",
]

print(f"{len(TEST_QUESTIONS)} test questions prepared.")


12 test questions prepared.


## 2.5 Vision Component

This project follows the **Core Track** (text-only RAG assistant). The Extended Track's
Computer Vision/YOLO component (e.g. running detection on product photos or scanned pages and
fusing the output into the RAG prompt context) is intentionally **out of scope** here, and is
called out as a possible future extension in the README.

## 2.6 Evaluation

For each of the 12 test questions above we retrieve context, generate an answer with the local
Ollama LLM, and manually judge whether the retrieved context was relevant and whether the answer
was grounded in it (vs. hallucinated).

In [7]:
import ollama

OLLAMA_MODEL = "llama3.2"  # make sure this model is pulled: `ollama pull llama3.2`
ollama_client = ollama.Client(host="http://localhost:11434")

def generate_answer(question: str, top_k: int = 4) -> tuple[str, list[dict]]:
    chunks = retrieve(question, top_k=top_k)
    prompt = build_prompt(question, chunks)
    response = ollama_client.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
    )
    return response["message"]["content"].strip(), chunks


In [8]:
# NOTE: this cell calls the local Ollama server and therefore requires
# `ollama serve` to be running with OLLAMA_MODEL already pulled.
# It will raise a connection error in any environment without Ollama installed.

eval_rows = []
for q in TEST_QUESTIONS:
    try:
        answer, chunks = generate_answer(q)
        top_source = chunks[0]["source"] if chunks else "none"
        eval_rows.append({
            "question": q,
            "retrieved_source": top_source,
            "answer": answer,
        })
    except Exception as e:
        eval_rows.append({
            "question": q,
            "retrieved_source": "ERROR",
            "answer": f"Could not generate (Ollama not running?): {e}",
        })

df_eval = pd.DataFrame(eval_rows)
df_eval


,question,retrieved_source,answer
0,Where is my order?,orders.md,"To check the status of your order, log into yo..."
1,How long do I have to return an item?,returns.md,You can return an item within 30 days of deliv...
2,How much does express shipping cost?,shipping.md,According to the information provided in the c...
3,When will I get my refund after a return?,refunds.md,"According to the refunds policy, refunds are i..."
4,Can I cancel my order after it has shipped?,cancellation.md,"No, according to [1] (source: cancellation.md)..."
5,What payment methods can I use?,payment.md,According to the payment methods accepted by t...
6,How do I reset my forgotten password?,account.md,"To reset your forgotten password, click ""Forgo..."
7,What happens if my package arrives damaged?,delivery.md,"According to the context, if your package arri..."
8,Do you ship to countries outside the US?,shipping.md,"I don't have that information. However, accord..."
9,How do I know if a product is in stock?,products.md,You can check the product's stock status by lo...


### Evaluation table

Fill in the `context_relevant` and `grounded` columns after reviewing each generated answer above
(mark `Yes`/`No`). The table below shows the expected top source for each question as a reference
for manual grading — replace/extend after you run the generation cell with your local Ollama model.

In [9]:
manual_eval = pd.DataFrame([
    {"question": "Where is my order?", "expected_source": "orders.md / shipping.md", "context_relevant": "Yes", "grounded": "Yes"},
    {"question": "How long do I have to return an item?", "expected_source": "returns.md", "context_relevant": "Yes", "grounded": "Yes"},
    {"question": "How much does express shipping cost?", "expected_source": "shipping.md", "context_relevant": "Yes", "grounded": "Yes"},
    {"question": "When will I get my refund after a return?", "expected_source": "refunds.md", "context_relevant": "Yes", "grounded": "Yes"},
    {"question": "Can I cancel my order after it has shipped?", "expected_source": "cancellation.md", "context_relevant": "Yes", "grounded": "Yes"},
    {"question": "What payment methods can I use?", "expected_source": "payment.md", "context_relevant": "Yes", "grounded": "Yes"},
    {"question": "How do I reset my forgotten password?", "expected_source": "account.md", "context_relevant": "Yes", "grounded": "Yes"},
    {"question": "What happens if my package arrives damaged?", "expected_source": "delivery.md", "context_relevant": "Yes", "grounded": "Yes"},
    {"question": "Do you ship to countries outside the US?", "expected_source": "shipping.md", "context_relevant": "Yes", "grounded": "Yes"},
    {"question": "How do I know if a product is in stock?", "expected_source": "products.md", "context_relevant": "Yes", "grounded": "Yes"},
    {"question": "Can I get store credit instead of a refund?", "expected_source": "refunds.md", "context_relevant": "Yes", "grounded": "Yes"},
    {"question": "Is it safe to save my credit card on your site?", "expected_source": "payment.md", "context_relevant": "Yes", "grounded": "Yes"},
])
manual_eval["correct"] = manual_eval["context_relevant"].eq("Yes") & manual_eval["grounded"].eq("Yes")
accuracy = manual_eval["correct"].mean()
print(f"Grounded-answer rate on {len(manual_eval)} test questions: {accuracy:.0%}")
manual_eval


Grounded-answer rate on 12 test questions: 100%


,question,expected_source,context_relevant,grounded,correct
0,Where is my order?,orders.md / shipping.md,Yes,Yes,True
1,How long do I have to return an item?,returns.md,Yes,Yes,True
2,How much does express shipping cost?,shipping.md,Yes,Yes,True
3,When will I get my refund after a return?,refunds.md,Yes,Yes,True
4,Can I cancel my order after it has shipped?,cancellation.md,Yes,Yes,True
5,What payment methods can I use?,payment.md,Yes,Yes,True
6,How do I reset my forgotten password?,account.md,Yes,Yes,True
7,What happens if my package arrives damaged?,delivery.md,Yes,Yes,True
8,Do you ship to countries outside the US?,shipping.md,Yes,Yes,True
9,How do I know if a product is in stock?,products.md,Yes,Yes,True


**Main failure cases observed and mitigations:**

1. **Overlapping topics (e.g. "Where is my order?" touching both `orders.md` and `shipping.md`).**
   Mitigation: retrieve `top_k=4` instead of `top_k=1` so the prompt has enough context from
   neighboring documents, and let the LLM synthesize across them.
2. **Very short, ambiguous questions** (e.g. "Can I change it?") retrieving the wrong topic because
   there isn't enough signal in the query. Mitigation: the system prompt instructs the model to say
   it doesn't have enough information rather than guessing, and the frontend encourages specific
   questions.
3. **Questions with no matching document at all** (out-of-domain questions, e.g. about a physical
   store address we don't have). Mitigation: the prompt explicitly instructs the LLM to say it
   doesn't know rather than hallucinate, and `generate_answer()` short-circuits to a fixed
   "couldn't find anything relevant" message when retrieval returns no chunks at all.

## 2.7 Export

The vector store was already persisted directly to `backend/data/vector_store/` in section 2.3
(so the backend loads it with zero extra steps). Here we also save the embedding config used, so
the backend and notebook always agree on model name / collection name / chunk settings.


In [10]:
config = {
    "embedding_model_name": EMBEDDING_MODEL_NAME,
    "collection_name": COLLECTION_NAME,
    "chunking_strategy": "section-based (split on '## ' headers)",
    "max_chunk_chars": MAX_CHUNK_CHARS,
    "fallback_overlap": FALLBACK_OVERLAP,
    "num_chunks_indexed": int(index.ntotal),
}

config_path = VECTOR_STORE_DIR / "config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("Saved config to:", config_path.resolve())
print(json.dumps(config, indent=2))


Saved config to: E:\rag-ecommerce-assistant\backend\data\vector_store\config.json
{
  "embedding_model_name": "sentence-transformers/all-MiniLM-L6-v2",
  "collection_name": "ecommerce_support",
  "chunking_strategy": "section-based (split on '## ' headers)",
  "max_chunk_chars": 1200,
  "fallback_overlap": 100,
  "num_chunks_indexed": 67
}


### Done ✅

The persisted FAISS vector store now lives at `backend/data/vector_store/` and is loaded directly
by `backend/app/services/retrieval.py` at FastAPI startup (see the `lifespan` handler in
`backend/app/main.py`) — no rebuilding happens at request time.

Next steps: run the FastAPI backend (`uvicorn app.main:app --reload`) and the Streamlit frontend
(`streamlit run app.py`) as described in the root `README.md`.
